<a href="https://colab.research.google.com/github/Qureshiii/PyTorch-Learning-Journey/blob/main/02_PyTorch_Autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.__version__)
import math

2.11.0+cu128


In [2]:
if torch.cuda.is_available():
  print('GPU is available!')
  print(f'Using GPU :{torch.cuda.get_device_name(0)}')
else:
  print('GPU is not available. Using CPU')

GPU is available!
Using GPU :Tesla T4


### Find out the derivative Without Autograd

In [44]:
def dy_dx(x):
  return 2*x

def dz_dx(x):
  return 2 * x * math.cos(x**2)


print(dy_dx(3))
print(dz_dx(3))

6
-5.466781571308061


### Now find the derivates With Autograd

In [27]:
x = torch.tensor(3.0, requires_grad=True)

In [28]:
y = x**2

In [29]:
print(x)
print(y)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)


In [30]:
# Now Find out Backward pass of y = x**2 is dy/dx
y.backward()
x.grad

tensor(6.)

In [45]:
# Now calculate this derivatives y = x**2 , z = sin(y)
x = torch.tensor(3.0, requires_grad=True)
y = x**2
z = torch.sin(y)

In [46]:
# Now Find out the derivatives of this equation
# dz/dx = dz/dy * dy/dx
z.backward()
x.grad

tensor(-5.4668)

### First Neural Network (Perceptron) without Autograd.

In [47]:
# Initialize input feature as a scalar tensor
x = torch.tensor(6.7) # Input feature

# Initialize true target label as a scalar tensor (0.0 for negative class)
y = torch.tensor(0.0) # true label(Binary)

# Initialize trainable weight parameter (initial slope)
w = torch.tensor(1.0) # weight

# Initialize trainable bias parameter (initial intercept)
b = torch.tensor(0.0) # Bias

In [48]:
# Binary cross-entropy loss for scalar

def binary_cross_entropy_loss(prediction,target):
  # Small value added to prevent underflow / calculating log(0) which yields -inf
  epsilon = 1e-8 # to prevent log(0)

  # Restrict prediction values to the range [epsilon, 1 - epsilon] for numerical stability
  prediction = torch.clamp(prediction,epsilon,1-epsilon)

  # Compute standard Binary Cross-Entropy formula: -[y*log(p) + (1-y)*log(1-p)]
  return - (target * torch.log(prediction) + (1-target) * torch.log(1-prediction))

In [49]:
# Forward pass

# Calculate linear combination of inputs (x), weights (w), and bias (b)
z = w * x + b

# Apply sigmoid activation to constrain output to probability range [0, 1]
y_pred = torch.sigmoid(z)

# Compute binary cross-entropy loss between predicted and target probabilities
loss = binary_cross_entropy_loss(y_pred,y)

In [50]:
# 1. Derivative of Binary Cross-Entropy Loss with respect to predicted output (y_pred)
# Loss function: L = -[y * log(y_pred) + (1 - y) * log(1 - y_pred)]
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. Derivative of the Sigmoid activation function (y_pred) with respect to weighted input (z)
# Activation function: y_pred = sigmoid(z) = 1 / (1 + e^-z)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. Local derivatives of weighted input (z) with respect to weights (w) and bias (b)
# Linear function: z = w*x + b
dz_dw = x
dz_db = 1

# 4. Total gradients with respect to parameters using the Chain Rule:
# dL/dw = (dL/dy_pred) * (dy_pred/dz) * (dz/dw)
dl_dw = dloss_dy_pred * dy_pred_dz * dz_dw

# dL/db = (dL/dy_pred) * (dy_pred/dz) * (dz/db)
dl_db = dloss_dy_pred * dy_pred_dz * dz_db

In [51]:
print(f'Manual Gradient of loss w.r.t wight(dw) : {dl_dw}')
print(f'Manual Gradient of loss w.r.t bias(db) : {dl_db}')

Manual Gradient of loss w.r.t wight(dw) : 6.691762447357178
Manual Gradient of loss w.r.t bias(db) : 0.998770534992218


### First Neural Network (Perceptron) with Autograd.

In [52]:
# Initialize input feature as a scalar tensor
x = torch.tensor(6.7) # Input feature

# Initialize true target label as a scalar tensor (0.0 for negative class)
y = torch.tensor(0.0) # true label(Binary)

# Initialize trainable weight parameter (initial slope)
w = torch.tensor(1.0,requires_grad=True) # weight

# Initialize trainable bias parameter (initial intercept)
b = torch.tensor(0.0,requires_grad=True) # Bias

In [56]:
# Forward propagation

z = w * x + b
y_pred = torch.sigmoid(z)
print(z)
print(y_pred)

tensor(6.7000, grad_fn=<AddBackward0>)
tensor(0.9988, grad_fn=<SigmoidBackward0>)


In [57]:
loss = binary_cross_entropy_loss(y_pred,y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [58]:
# Backward propagation
loss.backward()

In [59]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


### Using vectors

In [61]:
x = torch.tensor([1.0,2.0,3.0],requires_grad=True)

In [62]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [63]:
y.backward()

In [64]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

### Clearing Grad

In [70]:
x = torch.tensor(2.0,requires_grad=True)

In [83]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [79]:
y.backward()

In [82]:
x.grad

tensor(0.)

In [81]:
x.grad.zero_()

tensor(0.)

### Disable gradient tracking

 - 1. requires_grad_(False)

In [88]:
x = torch.tensor(2.0,requires_grad=True)

In [89]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [90]:
y.backward()

In [91]:
x.grad

tensor(4.)

In [92]:
x.requires_grad_(False)

tensor(2.)

 - 2. detach()

In [93]:
x = torch.tensor(2.0,requires_grad=True)

In [94]:
z = x.detach()

In [96]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

 - 3. torch.no_grad()

In [98]:
x = torch.tensor(2.0,requires_grad=True)

In [99]:
with torch.no_grad():
  y = x**2